# Reference Answer Builder

Use this notebook to inspect chunk text and manually build `reference_answers.json`. The benchmark should only use queries whose relevant chunk IDs have been reviewed here.

## Workflow

1. Add PDFs under `data/pdfs/<Company_Name>/`.
2. Run `python pipeline.py extract` so `data/interim/chunks/*_chunks.jsonl` exists.
3. Use the search/inspect cells below to find evidence chunks.
4. Edit `reference_answers.json` manually once you have verified the chunk IDs.
5. Run the validation cell before benchmarking.

In [ ]:
import json
import re
import sys
import unicodedata
from pathlib import Path

from IPython.display import Markdown, display
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pipeline.py").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation.paths import REFERENCE_ANSWERS

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)
CHUNKS_DIR = REPO_ROOT / "data/interim/chunks"
REFERENCE_PATH = REFERENCE_ANSWERS
EXPORT_DIR = REPO_ROOT / "reference_review_exports"

In [ ]:
def load_chunks(chunks_dir=CHUNKS_DIR):
    rows = []
    for path in sorted(chunks_dir.glob("*_chunks.jsonl")):
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                chunk = json.loads(line)
                company = chunk["company"]
                source = chunk["source_file"]
                chunk_id = f"{company.replace(' ', '_')}_{source.replace(' ', '_')}_{chunk['id']}"
                text = chunk["text"]
                rows.append(
                    {
                        "chunk_id": chunk_id,
                        "company": company,
                        "source_file": source,
                        "pages": chunk.get("pages", []),
                        "text": text,
                        "preview": text.replace("\n", " ")[:500],
                    }
                )
    return pd.DataFrame(rows)

chunks = load_chunks()
print(f"Loaded {len(chunks)} chunks from {chunks['source_file'].nunique() if not chunks.empty else 0} document(s)")
display(chunks[["chunk_id", "company", "source_file", "pages", "preview"]].head(10))

## Full-Text Reading Helpers

Use these helpers instead of relying on DataFrame previews. They print the complete chunk text in readable Markdown.

In [ ]:
def chunk_markdown(row):
    return (
        f"### `{row['chunk_id']}`\n\n"
        f"**Company:** {row['company']}  \n"
        f"**Source:** {row['source_file']}  \n"
        f"**Pages:** {row['pages']}\n\n"
        "```text\n"
        f"{row['text']}\n"
        "```\n"
    )


def show_chunk(chunk_id):
    row = chunks.loc[chunks["chunk_id"] == chunk_id]
    if row.empty:
        print(f"Chunk ID not found: {chunk_id}")
        return
    display(Markdown(chunk_markdown(row.iloc[0])))


def show_chunks(chunk_ids):
    for chunk_id in chunk_ids:
        show_chunk(chunk_id)


def select_chunks(company=None, source_file=None, max_chunks=None):
    subset = chunks.copy()
    if company is not None:
        subset = subset[subset["company"] == company]
    if source_file is not None:
        subset = subset[subset["source_file"] == source_file]
    if max_chunks is not None:
        subset = subset.head(max_chunks)
    return subset


def show_document(company=None, source_file=None, max_chunks=None):
    subset = select_chunks(company=company, source_file=source_file, max_chunks=max_chunks)

    for _, row in subset.iterrows():
        display(Markdown(chunk_markdown(row)))


def export_chunks_to_markdown(chunk_ids, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    selected = chunks[chunks["chunk_id"].isin(chunk_ids)]
    body = "\n\n".join(chunk_markdown(row) for _, row in selected.iterrows())
    path.write_text(body, encoding="utf-8")
    print(f"Wrote {len(selected)} chunk(s) to {path}")


def export_document_to_markdown(path, company=None, source_file=None, max_chunks=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    selected = select_chunks(company=company, source_file=source_file, max_chunks=max_chunks)
    body = "\n\n".join(chunk_markdown(row) for _, row in selected.iterrows())
    path.write_text(body, encoding="utf-8")
    print(f"Wrote {len(selected)} chunk(s) to {path}")

## Normalized Search Helpers

PDF extraction can split words across lines, keep soft hyphens, or replace bullets. Use `search_chunks()` when exact string search misses text you can see in the PDF.

In [ ]:
def normalize_for_search(value):
    text = unicodedata.normalize("NFKC", str(value)).lower()
    text = re.sub(r"[\x00-\x1f\x7f]", "", text)
    text = re.sub(r"[\u2010\u2011\u2012\u2013\u2014\u2212]", "-", text)
    text = re.sub(r"-\s+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def search_chunks(terms, company=None, source_file=None, page=None, limit=30):
    subset = select_chunks(company=company, source_file=source_file).copy()
    if page is not None:
        subset = subset[subset["pages"].apply(lambda pages: page in pages)]

    subset["search_text"] = subset["text"].map(normalize_for_search)
    mask = pd.Series(True, index=subset.index)
    for term in terms:
        mask &= subset["search_text"].str.contains(normalize_for_search(term), regex=False, na=False)

    result = subset.loc[mask, ["chunk_id", "company", "source_file", "pages", "preview"]]
    print(f"Matches: {len(result)}")
    display(result.head(limit))
    return result

## Document Coverage

Check which documents are available before drafting questions.

In [ ]:
display(
    chunks.groupby(["company", "source_file"], as_index=False)
    .agg(chunks=("chunk_id", "count"), first_pages=("pages", "first"))
)

## Read A Whole Document As Chunks

Use this when you want to move through a document chunk by chunk. Start with `max_chunks=5`, then remove it when you are ready to read the whole document.

In [ ]:
show_document(
    company="AGL",
    source_file="Carbon Constrained Future",
    max_chunks=5,
)

## Search Extracted PDF Text Robustly

Use this version when copied PDF text does not match because of line wrapping, hyphenation, or bullet formatting.

In [ ]:
matches = search_chunks(
    ["Clean Energy Vision", "fossil fuel generation", "net-zero CO2"],
    company="Alliant",
    source_file="CDP 2022",
)

show_chunks(matches["chunk_id"].head(5).tolist())

## Search Chunks

Change `SEARCH_TERMS` and rerun. Use terms from the question you want to label.

In [ ]:
SEARCH_TERMS = ["emissions", "target"]

mask = pd.Series(True, index=chunks.index)
for term in SEARCH_TERMS:
    mask &= chunks["text"].str.contains(term, case=False, regex=False)

matches = chunks.loc[mask, ["chunk_id", "company", "source_file", "pages", "preview"]]
print(f"Matches: {len(matches)}")
display(matches.head(30))

## Read Search Matches In Full

After running the search cell, use this to read the full text for the first few matches.

In [ ]:
show_chunks(matches["chunk_id"].head(5).tolist())

## Inspect Full Chunk Text

Paste a candidate chunk ID below to read the complete evidence text. This does not truncate like the preview table.

In [ ]:
CHUNK_ID = "AGL_Carbon_Constrained_Future_elem_0012"

show_chunk(CHUNK_ID)

## Inspect Multiple Evidence Chunks

Use this to review all chunks you plan to list as relevant for one query.

In [ ]:
CANDIDATE_IDS = [
    "AGL_Carbon_Constrained_Future_elem_0012",
    "AGL_Carbon_Constrained_Future_elem_0015",
]

show_chunks(CANDIDATE_IDS)

## Export Chunks For Reading

If the notebook display is still awkward, export candidate chunks to a Markdown file and open it in VS Code.

In [ ]:
# export_chunks_to_markdown(
#     CANDIDATE_IDS,
#     EXPORT_DIR / "agl_emissions_targets_chunks.md",
# )

## Export A Whole Document For Reading

This writes the selected chunks to a Markdown file, which is usually easier to read and annotate in VS Code than notebook output.

In [ ]:
# export_document_to_markdown(
#     EXPORT_DIR / "agl_carbon_constrained_future_all_chunks.md",
#     company="AGL",
#     source_file="Carbon Constrained Future",
# )

## Draft A Reference Entry

This cell only prints a JSON object. It does not edit `reference_answers.json`.

In [ ]:
QUERY = "What are AGL's emissions targets?"
RELEVANT_IDS = [
    "AGL_Carbon_Constrained_Future_elem_0012",
    "AGL_Carbon_Constrained_Future_elem_0015",
]

draft = {"query": QUERY, "relevant_ids": RELEVANT_IDS}
print(json.dumps(draft, indent=2))

## Validate `reference_answers.json`

Run this after editing the JSON file. Missing IDs must be fixed before benchmarking.

In [ ]:
reference_answers = json.loads(REFERENCE_PATH.read_text(encoding="utf-8"))
known_ids = set(chunks["chunk_id"])
missing = []

for query_number, entry in enumerate(reference_answers, start=1):
    for chunk_id in entry["relevant_ids"]:
        if chunk_id not in known_ids:
            missing.append({"query_number": query_number, "missing_chunk_id": chunk_id})

print(f"Reference queries: {len(reference_answers)}")
print(f"Missing IDs: {len(missing)}")
display(pd.DataFrame(missing))
display(pd.DataFrame(reference_answers))